# Semaine 4 — Multi-Agent & MCP (Python pur)

**Objectif :** Implémenter une simulation multi-agent et un minimal MCP-like exchange (mock + API adaptations).

## Installation
```bash
pip install requests networkx python-dotenv
```

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
print('Ready')

## Mock multi-agent simulation
Trois agents : researcher, planner, executor. Ils communiquent via un bus central (simple Python queue).

In [ ]:
from collections import deque
import json

class Agent:
    def __init__(self, name, role):
        self.name = name
        self.role = role
        self.inbox = deque()
    def receive(self, msg):
        self.inbox.append(msg)
    def step(self):
        outs = []
        while self.inbox:
            m = self.inbox.popleft()
            content = m.get('content')
            # simple role-based behavior
            if self.role == 'researcher':
                outs.append({'to':'planner','content':f'findings about {content}'})
            elif self.role == 'planner':
                outs.append({'to':'executor','content':f'plan for {content}'})
            elif self.role == 'executor':
                outs.append({'to':'user','content':f'executed {content}'})
        return outs

# instantiate
r = Agent('researcher','researcher')
p = Agent('planner','planner')
e = Agent('executor','executor')

# start
r.receive({'from':'user','content':'Organiser un atelier produit la semaine prochaine'})
for _ in range(3):
    for ag in [r,p,e]:
        for out in ag.step():
            if out['to']=='planner': p.receive({'from':ag.name,'content':out['content']})
            if out['to']=='executor': e.receive({'from':ag.name,'content':out['content']})
            if out['to']=='user': print('[USER]', out['content'])

print('Multi-agent simulation complete')

## Minimal MCP-like message format
On définit un schéma simple JSON pour échanger contexte entre agents (id, type, payload, provenance, permissions).

In [ ]:
# Example MCP-like message
mcp_msg = {
    'id':'msg-001',
    'type':'context_update',
    'from':'researcher',
    'to':'planner',
    'payload':{'summary':'Top-3 venues found'},
    'permissions':{'read':['planner','executor'],'write':['researcher']}
}
print('MCP-like message:', json.dumps(mcp_msg, indent=2, ensure_ascii=False))

## Security & filtering
Exemple de policy en Python pour vérifier si un agent peut lire un message.

In [ ]:
def can_read(agent_name, mcp_message):
    allowed = mcp_message.get('permissions', {}).get('read', [])
    return agent_name in allowed

print('planner can read?', can_read('planner', mcp_msg))
print('user can read?', can_read('user', mcp_msg))

## Bench and metrics
Collect basic metrics: iterations, latency (mock), number of messages exchanged.

In [ ]:
# simple metrics while running scenario
logs = []
# run small scenario and log
r.receive({'from':'user','content':'Verifier disponibilite salle'})
for i in range(3):
    for ag in [r,p,e]:
        outs = ag.step()
        for o in outs:
            logs.append({'from':ag.name,'to':o['to'],'content':o['content']})
            if o['to']=='planner': p.receive({'from':ag.name,'content':o['content']})
            if o['to']=='executor': e.receive({'from':ag.name,'content':o['content']})
            if o['to']=='user': print('[USER]', o['content'])

print('Logs:')
print(json.dumps(logs, indent=2, ensure_ascii=False))

Fin du notebook S4 Multi-Agent Python (mock + API adaptations)